# Mutual Fund Analytics — Day 1: Data Ingestion & EDA

**Objective:** Load all 10 provided CSV datasets, fetch live NAV data from mfapi.in, and validate AMFI scheme codes.

---

In [ ]:
from pathlib import Path
import json
import time

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import requests

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.4f}'.format)
plt.style.use('seaborn-v0_8-darkgrid')

ROOT = Path('..').resolve()
RAW_DIR  = ROOT / 'data' / 'raw'
PROC_DIR = ROOT / 'data' / 'processed'
RPT_DIR  = ROOT / 'reports'
PROC_DIR.mkdir(parents=True, exist_ok=True)
RPT_DIR.mkdir(parents=True, exist_ok=True)

print('Root:', ROOT)
print('Raw CSV files found:', len(list(RAW_DIR.rglob('*.csv'))))

## 1. Load All 10 CSV Datasets
Print `.shape`, `.dtypes`, and `.head()` for each, and flag anomalies.

In [ ]:
csv_paths = sorted(p for p in RAW_DIR.rglob('*.csv') if not p.stem.startswith('nav_') and not p.stem.startswith('live_'))
datasets = {}

for path in csv_paths:
    df = pd.read_csv(path, low_memory=False)
    datasets[path.stem] = df
    print('=' * 80)
    print(f'FILE  : {path.name}')
    print(f'SHAPE : {df.shape}')
    print('\nDTYPES:')
    print(df.dtypes)
    print('\nHEAD:')
    display(df.head())
    missing = df.isna().sum().sum()
    dupes   = df.duplicated().sum()
    print(f'\nANOMALIES → missing cells: {missing}, duplicate rows: {dupes}')
    print()

## 2. Fund Master Exploration
Unique fund houses, categories, sub-categories, risk grades, and AMFI scheme code structure.

In [ ]:
# Find fund master dataset
master_key = next((k for k in datasets if 'fund_master' in k.lower()), None)
if master_key:
    master = datasets[master_key]
    print('Columns:', list(master.columns))
    display(master.head())
else:
    print('fund_master file not found in datasets')

In [ ]:
if master_key:
    master = datasets[master_key]
    for col in master.columns:
        lowcol = col.lower().replace(' ', '_')
        if any(kw in lowcol for kw in ['fund_house', 'amc', 'fundhouse']):
            print(f'\nUnique Fund Houses [{col}] ({master[col].nunique()}):')
            print(sorted(master[col].dropna().unique().tolist()))
        elif any(kw in lowcol for kw in ['category', 'scheme_category']):
            print(f'\nUnique Categories [{col}] ({master[col].nunique()}):')
            print(sorted(master[col].dropna().unique().tolist()))
        elif any(kw in lowcol for kw in ['sub_category', 'sub-category', 'scheme_type']):
            print(f'\nUnique Sub-Categories [{col}] ({master[col].nunique()}):')
            print(sorted(master[col].dropna().unique().tolist()))
        elif any(kw in lowcol for kw in ['risk', 'riskometer', 'risk_grade']):
            print(f'\nUnique Risk Grades [{col}] ({master[col].nunique()}):')
            print(sorted(master[col].dropna().unique().tolist()))

## 3. Live NAV Fetch — mfapi.in
Fetch HDFC Top 100 Direct and 5 key large-cap schemes.

In [ ]:
BASE_URL = 'https://api.mfapi.in/mf/'
SCHEMES = {
    '125497': 'HDFC_Top_100_Direct',
    '119551': 'SBI_Bluechip',
    '120503': 'ICICI_Bluechip',
    '118632': 'Nippon_Large_Cap',
    '119092': 'Axis_Bluechip',
    '120841': 'Kotak_Bluechip',
}

live_frames = []
with requests.Session() as sess:
    sess.headers.update({'User-Agent': 'MutualFundAnalytics/1.0'})
    for code, label in SCHEMES.items():
        try:
            r = sess.get(f'{BASE_URL}{code}', timeout=30)
            r.raise_for_status()
            payload = r.json()
            meta = payload.get('meta', {})
            df = pd.DataFrame(payload.get('data', []))
            df['scheme_code'] = code
            df['scheme_name'] = meta.get('scheme_name', label)
            df['fund_house']   = meta.get('fund_house')
            if 'nav' in df.columns:
                df['nav'] = pd.to_numeric(df['nav'], errors='coerce')
            if 'date' in df.columns:
                df['date'] = pd.to_datetime(df['date'], format='%d-%m-%Y', errors='coerce')
            live_frames.append(df)
            latest = df.loc[df['date'].idxmax()] if 'date' in df.columns else None
            print(f'✓ {label} ({code}): {len(df):,} rows | Latest NAV: {latest["nav"] if latest is not None else "N/A"} on {latest["date"].date() if latest is not None else "N/A"}')
        except Exception as e:
            print(f'✗ {label} ({code}): {e}')
        time.sleep(0.25)

live_nav = pd.concat(live_frames, ignore_index=True)
print(f'\nTotal live NAV rows: {len(live_nav):,}')
display(live_nav.head(10))

## 4. AMFI Scheme Code Validation
Confirm every code in fund_master exists in nav_history.

In [ ]:
history_key = next((k for k in datasets if 'nav_history' in k.lower()), None)

if master_key and history_key:
    master  = datasets[master_key]
    history = datasets[history_key]

    # Infer scheme_code column robustly
    def find_code_col(df):
        for c in df.columns:
            if 'scheme' in c.lower() and 'code' in c.lower():
                return c
            if 'amfi' in c.lower() and 'code' in c.lower():
                return c
        for c in df.columns:
            if c.lower() == 'code':
                return c
        return None

    mc = find_code_col(master)
    hc = find_code_col(history)
    print(f'Master code column : {mc}')
    print(f'History code column: {hc}')

    if mc and hc:
        master_codes  = set(master[mc].dropna().astype(str).str.strip().str.replace(r'\.0$', '', regex=True))
        history_codes = set(history[hc].dropna().astype(str).str.strip().str.replace(r'\.0$', '', regex=True))
        missing = sorted(master_codes - history_codes)
        matched = sorted(master_codes & history_codes)

        print(f'\nMaster unique codes : {len(master_codes):,}')
        print(f'History unique codes: {len(history_codes):,}')
        print(f'Matched codes       : {len(matched):,}')
        print(f'Missing from history: {len(missing):,}')
        if missing:
            print('Missing codes sample:', missing[:10])
        print(f'Validation passed   : {len(missing) == 0}')
    else:
        print('Could not infer scheme code columns.')
else:
    print('fund_master or nav_history not found.')

## 5. Quick Visualizations — NAV Trend

In [ ]:
fig = px.line(
    live_nav.sort_values('date'),
    x='date', y='nav', color='scheme_name',
    title='Historical NAV Comparison — 6 Large-Cap Schemes',
    labels={'nav': 'NAV (₹)', 'date': 'Date', 'scheme_name': 'Scheme'},
    template='plotly_dark'
)
fig.update_layout(height=500)
fig.show()

In [ ]:
# Latest NAV snapshot bar chart
latest_navs = (
    live_nav.sort_values('date')
    .groupby('scheme_name', as_index=False)
    .last()[['scheme_name', 'date', 'nav']]
)
fig2 = px.bar(
    latest_navs, x='scheme_name', y='nav', color='scheme_name',
    title='Latest NAV Snapshot',
    labels={'nav': 'NAV (₹)', 'scheme_name': 'Scheme'},
    template='plotly_dark', text_auto='.2f'
)
fig2.update_layout(showlegend=False, height=400)
fig2.show()

## 6. Summary Report

In [ ]:
summary = {
    'csv_files_loaded': len(datasets),
    'live_schemes_fetched': len(SCHEMES),
    'live_nav_rows': len(live_nav),
    'dataset_shapes': {k: list(v.shape) for k, v in datasets.items()}
}
print(json.dumps(summary, indent=2))
print('\n✅ Day 1 data ingestion complete!')